# Larger-model baseline: Qwen2.5-7B vs Claude

**This notebook evaluates existing models; it does not fine-tune or publish weights.** Use **Runtime → Change runtime type → T4 GPU**, then **Run all**. Upload the same Amplify PDF when prompted. All 41 distinct Amplify writings, your two style comparisons, your six-category rubric and the 98 OpenStax annotation references are included. Repeated OpenStax inputs share one inference while retaining every reference.

New material: a deterministic balanced challenge from ASAP 2.0 (default: up to one essay for each of seven prompt × six score cells, maximum 42), with original source readings and the native holistic 1–6 rubric. Also collect FOXGLOVE's 69 essays with human feedback and use 14 cases for qualitative review. AI-written FOXGLOVE feedback is not a reference target. Public corpora are downloaded in your runtime, not embedded in the notebook. Protected official ASAP test data is not downloaded. Demographics are not sent to models.

Amplify scores stay on their original 1–4 criteria. Your rubric has separate qualitative probes because six-category teacher ratings are unavailable. Existing material is development data, not a new independent test. ASAP cases are new to this experiment; pretraining overlap is unknown. Scanned source PDFs use OCR, which requires proofreading. Reference comments are alternatives to compare against, not exact-match targets.

Qwen runs locally on the Colab GPU without an API key. The optional Claude comparison asks for a key with hidden input and uses your API account; leave it blank to skip. Keys are not exported. The default is one deterministic response per case. Do not change prompts after reviewing the challenge labels and call a rerun a fresh test.


In [ ]:
import pathlib, subprocess, sys, os, io, zipfile, base64, hashlib, json
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before model downloads. No model training or inference started.")
print("GPU:", torch.cuda.get_device_name(0))
repo=pathlib.Path("/content/writing-baseline-v2") if pathlib.Path("/content").exists() else pathlib.Path.cwd()/"writing-baseline-v2"
source_zip=base64.b64decode('...')
source_sha256='013cabb69f7de2652fff3a5dbede05b87af921ac09ee22d6d60c26d4ae16a988'
assert hashlib.sha256(source_zip).hexdigest()==source_sha256
repo.mkdir(exist_ok=True)
identity=repo/"code-snapshot.json"
if identity.exists() and json.loads(identity.read_text())["sha256"]!=source_sha256:
    raise RuntimeError("This directory contains a different notebook revision. Use a fresh runtime.")
with zipfile.ZipFile(io.BytesIO(source_zip)) as archive:
    for name in archive.namelist():
        if not (repo/name).resolve().is_relative_to(repo.resolve()):raise RuntimeError("Invalid source archive")
    archive.extractall(repo)
os.chdir(repo)
identity.write_text(json.dumps({"sha256":source_sha256},indent=2))
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-baseline.txt"],check=True)
import shutil
if not shutil.which("pdftotext") or not shutil.which("tesseract"):
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","poppler-utils","tesseract-ocr"],check=True)
print("Inference dependencies and PDF/OCR tools ready.")


## Load your supplied material

Upload **Student_Work_Collection__Essay_Scoring.pdf**. The notebook verifies its checksum and uses original page versions to resolve repeated-page mistakes. The unchanged style preferences already in your repository are fetched separately with a checksum; no uploaded writing or preference examples are embedded in this notebook.


In [ ]:
from google.colab import files
from urllib.request import urlopen
from model_work.amplify_native import PDF_SHA256
uploaded=files.upload()
pdfs=[pathlib.Path(name).resolve() for name,data in uploaded.items() if hashlib.sha256(data).hexdigest()==PDF_SHA256]
if len(pdfs)!=1:raise RuntimeError("Upload the same reviewed 120-page Amplify PDF.")
amplify_pdf=pdfs[0]
preference_url="https://raw.githubusercontent.com/sfstast/essaygrader/8d05ea2be59aa920c1e04216566109835eab83ad/model_work/examples/user_style_preferences.json"
with urlopen(preference_url,timeout=30) as response:preference_bytes=response.read()
if hashlib.sha256(preference_bytes).hexdigest()!="f69be2d4e472a11ebf6f20cbd88d0f65f5adab380f2dcac043c2c13ed177a271":raise RuntimeError("Preference-source checksum mismatch")
preferences=repo/"user-style-preferences.json";preferences.write_bytes(preference_bytes)
print("Amplify PDF and two supplied preference pairs verified.")


## Prepare the fixed evaluation cases

This stage may take several minutes because five source readings need OCR. All 17,307 public ASAP training essays and 696 human FOXGLOVE comments are collected locally, but only the fixed challenge sample is sent to the models. Source/label provenance is retained. No demographic fields become model inputs. Do not send the entire corpus through a paid API by default.


In [ ]:
from model_work import openstax_annotations
from model_work.baseline_data import prepare
ASAP_PER_PROMPT_SCORE=1
FOXGLOVE_REVIEW_CASES=14
source_dir=repo/"data/openstax-original"
annotation_dir=repo/"data/openstax-prepared"
if not (source_dir/"manifest.json").exists():openstax_annotations.download(source_dir)
if not (annotation_dir/"manifest.json").exists():openstax_annotations.prepare(source_dir,annotation_dir)
dataset=repo/"data/baseline-cases"
if (dataset/"manifest.json").exists():
    manifest=json.loads((dataset/"manifest.json").read_text())
    if manifest.get("asap_per_prompt_score")!=ASAP_PER_PROMPT_SCORE or manifest.get("foxglove_review_cases")!=FOXGLOVE_REVIEW_CASES:raise RuntimeError("Sampling settings changed; start a separate benchmark directory.")
    print("Reusing previously fixed cases.")
else:
    if dataset.exists():raise RuntimeError("An interrupted preparation directory exists. Save logs and start a fresh runtime.")
    manifest=prepare(amplify_pdf,annotation_dir,preferences,dataset,ASAP_PER_PROMPT_SCORE,FOXGLOVE_REVIEW_CASES)
from model_work.baseline_runner import load_cases,run,write_review
cases=load_cases(dataset)
print(json.dumps({k:manifest[k] for k in ("unique_inference_cases","source_reference_records","counts_by_track")},indent=2))
results=repo/"results";results.mkdir(exist_ok=True)


## Evaluate Qwen2.5-7B without fine-tuning

Loads the official instruction model in 4-bit form. Resolve and record the exact model revision; verify its model card and license before loading. Every case receives the same fixed prompt used for Claude. Inputs exceeding the configured limit are reported as failures, never silently truncated. Predictions save after every case, and an unchanged run can resume. Model generation can take a while on a free T4.


In [ ]:
from huggingface_hub import HfApi,hf_hub_download
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
import re,time
MODEL_ID="Qwen/Qwen2.5-7B-Instruct"
MAX_INPUT_TOKENS=8192
MAX_NEW_TOKENS=256
model_file=results/"qwen-model.json"
if model_file.exists():revision=json.loads(model_file.read_text())["revision"]
else:revision=HfApi().model_info(MODEL_ID).sha
if not re.fullmatch(r"[0-9a-f]{40}",revision):raise RuntimeError("Unexpected model revision")
card=pathlib.Path(hf_hub_download(MODEL_ID,"README.md",revision=revision)).read_text()
license_path=pathlib.Path(hf_hub_download(MODEL_ID,"LICENSE",revision=revision))
if not re.search(r"^license:\s*apache-2\.0\s*$",card,re.M):raise RuntimeError("Unexpected pinned model license declaration")
if not re.search(r"Apache License.*?Version 2\.0",license_path.read_text(),re.S):raise RuntimeError("Unexpected model license text")
model_file.write_text(json.dumps({"model":MODEL_ID,"revision":revision,"weight_license":"Apache-2.0","license_sha256":hashlib.sha256(license_path.read_bytes()).hexdigest()},indent=2))
shutil.copy2(license_path,results/"QWEN_LICENSE.txt")
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=revision,trust_remote_code=False)
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=revision,trust_remote_code=False,
    quantization_config=quant,device_map="auto",torch_dtype=torch.float16,attn_implementation="sdpa")
model.eval()
class InputTooLongError(ValueError):pass
def qwen_response(messages):
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    tokens=tokenizer(text,return_tensors="pt",add_special_tokens=False,truncation=False)
    if tokens["input_ids"].shape[1]>MAX_INPUT_TOKENS:raise InputTooLongError("No truncation allowed")
    tokens={k:v.to(model.device) for k,v in tokens.items()}
    with torch.inference_mode():
        generated=model.generate(**tokens,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(generated[0,tokens["input_ids"].shape[1]:],skip_special_tokens=True)
qwen_metrics=run(dataset,results,"qwen_7b_baseline",qwen_response,
    {"model":MODEL_ID,"revision":revision,"quantization":"4-bit NF4","max_input_tokens":MAX_INPUT_TOKENS,"max_new_tokens":MAX_NEW_TOKENS,"do_sample":False})
print(json.dumps(qwen_metrics,indent=2))


## Save the Qwen result before optional Claude calls

This downloads predictions and metrics, not model weights. Feedback accuracy still needs human review. A complete result means every case returned the expected format; it does not establish grading quality. Watch exact agreement, native-scale error, score histograms, invalid responses, absent quotes, and the fixed-score-3 baseline.


In [ ]:
def package_results(label):
    write_review(dataset,results)
    for filename in ("manifest.json","ATTRIBUTION.txt"):
        shutil.copy2(dataset/filename,results/filename)
    shutil.copy2(repo/"code-snapshot.json",results/"code-snapshot.json")
    import importlib.metadata
    versions={name:importlib.metadata.version(name) for name in ("torch","transformers","accelerate","bitsandbytes","anthropic")}
    versions["tesseract"]=subprocess.check_output(["tesseract","--version"],text=True).splitlines()[0]
    (results/"runtime-versions.json").write_text(json.dumps(versions,indent=2))
    for source in ("asap2","foxglove"):
        shutil.copy2(dataset/source/"provenance.json",results/(source+"-provenance.json"))
    shutil.copy2(dataset/"asap2/source-extraction-methods.json",results/"source-extraction-methods.json")
    (results/"README.txt").write_text("Inference-only baseline comparison. No training. Native rubric scales must not be combined. Inputs and references in the human-review file contain student writing; keep the downloaded result private. OCR readings need proofreading. No model is automatically deployed.\n")
    archive=shutil.make_archive(str(repo/label),"zip",results)
    files.download(archive)
    return archive
qwen_archive=package_results("qwen-7b-baseline-results")


## Optional Claude comparison on the identical cases

Claude calls use your Anthropic API account. Set `RUN_CLAUDE=False`, or leave the hidden key input blank, to skip. No API key is included in saved files. Comparison summaries explicitly show a skipped Claude run; Qwen-only results are not described as a Claude comparison.


In [ ]:
RUN_CLAUDE=True
CLAUDE_MODEL="claude-haiku-4-5"
if RUN_CLAUDE:
    from getpass import getpass
    claude_key=getpass("Anthropic API key (blank skips Claude): ").strip()
    if claude_key:
        import anthropic
        client=anthropic.Anthropic(api_key=claude_key,timeout=120,max_retries=2)
        def claude_response(messages):
            answer=client.messages.create(model=CLAUDE_MODEL,max_tokens=MAX_NEW_TOKENS,temperature=0,
                system=messages[0]["content"],messages=messages[1:])
            return "".join(block.text for block in answer.content if getattr(block,"type",None)=="text")
        try:
            claude_metrics=run(dataset,results,"claude_baseline",claude_response,
                {"model":CLAUDE_MODEL,"max_new_tokens":MAX_NEW_TOKENS,"temperature":0})
            print(json.dumps(claude_metrics,indent=2))
        finally:
            client.close();claude_key=""
    else:
        (results/"claude-skipped.json").write_text(json.dumps({"status":"skipped","reason":"No API key supplied"}))
else:
    (results/"claude-skipped.json").write_text(json.dumps({"status":"skipped","reason":"RUN_CLAUDE=False"}))
final_archive=package_results("larger-model-baseline-results")
print("Upload the final results ZIP back to the chat. No fine-tuning or deployment ran.")
